# NUTS Algorithm in Practice

## Learning Objectives
By completing this notebook, you will:
1. Understand how the No-U-Turn Sampler (NUTS) works internally
2. Learn to configure NUTS parameters for optimal performance
3. Diagnose and resolve common sampling issues
4. Apply NUTS to commodity price models with challenging posteriors
5. Compare NUTS with other sampling algorithms

## Prerequisites
- Understanding of MCMC basics (Module 1)
- Familiarity with Hamiltonian Monte Carlo
- Experience with PyMC models

## Estimated Time: 60 minutes

---

In [ ]:
learning_objectives(["Understand how the No-U-Turn Sampler (NUTS) works internally", "Learn to configure NUTS parameters for optimal performance", "Diagnose and resolve common sampling issues", "Apply NUTS to commodity price models with challenging posteriors", "Compare NUTS with other sampling algorithms", "Understanding of MCMC basics (Module 1)"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## 1. Setup and Imports

We'll import PyMC, ArviZ for diagnostics, and utilities for working with commodity data.

In [ ]:
# Purpose: Import required libraries for NUTS sampling and diagnostics
# Key Concept: ArviZ provides comprehensive diagnostic tools for MCMC

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats
import warnings

# Configuration
warnings.filterwarnings('ignore')
np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## 2. Understanding NUTS: The Algorithm

### What is NUTS?

The **No-U-Turn Sampler (NUTS)** is an adaptive variant of Hamiltonian Monte Carlo (HMC) that:
- Automatically tunes the number of leapfrog steps
- Eliminates the need to manually set trajectory length
- Uses a criterion based on "U-turns" to stop simulating

### Key Components

1. **Hamiltonian dynamics**: Uses gradient information to propose moves
2. **Leapfrog integrator**: Simulates Hamiltonian dynamics discretely
3. **No-U-turn criterion**: Stops when trajectory starts doubling back
4. **Dual averaging**: Adaptively tunes step size during warmup

Let's visualize how NUTS explores a 2D posterior.

In [ ]:
# Purpose: Demonstrate NUTS sampling on a simple 2D model
# Key Concept: NUTS efficiently explores correlated posteriors

# Create a model with correlated parameters (common in commodity models)
with pm.Model() as correlated_model:
    # True values: mu1=5, mu2=10, correlation=0.8
    mu = pm.MvNormal('mu', 
                     mu=[5, 10], 
                     cov=[[1.0, 0.8], [0.8, 1.0]], 
                     shape=2)
    
    # Generate synthetic observations
    true_mu = [5, 10]
    obs_data = np.random.multivariate_normal(
        true_mu, 
        [[1.0, 0.8], [0.8, 1.0]], 
        size=50
    )
    
    # Likelihood
    obs = pm.MvNormal('obs', 
                      mu=mu, 
                      cov=[[1.0, 0.8], [0.8, 1.0]], 
                      observed=obs_data)
    
    # Sample with NUTS (default in PyMC)
    trace_nuts = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        random_seed=42,
        return_inferencedata=True
    )

# Visualize posterior
az.plot_pair(
    trace_nuts,
    var_names=['mu'],
    divergences=True,
    figsize=(10, 8)
)
plt.suptitle('NUTS Posterior: Correlated Parameters', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print("\nNotice how NUTS efficiently explores the correlation structure.")

## 3. NUTS Parameters and Tuning

### Key Parameters

1. **`target_accept`**: Target acceptance probability (default: 0.8)
   - Higher values → smaller steps → slower but more accurate
   - Lower values → larger steps → faster but may have divergences

2. **`max_treedepth`**: Maximum tree depth (default: 10)
   - Controls maximum number of leapfrog steps (2^max_treedepth)
   - Increase if you see "max_treedepth" warnings

3. **Step size (ε)**: Automatically tuned during warmup
   - Can be manually set if needed

Let's see how these parameters affect sampling.

In [ ]:
# Purpose: Demonstrate effect of target_accept on sampling
# Key Concept: Higher target_accept reduces divergences but slows sampling

# Create a challenging model: stochastic volatility
# Generate synthetic crude oil returns with time-varying volatility
np.random.seed(123)
n_obs = 200
true_log_vol = np.zeros(n_obs)
true_log_vol[0] = 0.0
for t in range(1, n_obs):
    true_log_vol[t] = 0.95 * true_log_vol[t-1] + np.random.randn() * 0.2

returns = np.random.randn(n_obs) * np.exp(true_log_vol / 2)

# Fit with different target_accept values
results = {}

for target_accept in [0.8, 0.95]:
    print(f"\nSampling with target_accept={target_accept}...")
    
    with pm.Model() as sv_model:
        # Hyperparameters
        sigma_vol = pm.Exponential('sigma_vol', 1/0.5)
        rho = pm.Beta('rho', alpha=20, beta=2)  # High persistence
        
        # Log-volatility process
        log_vol = pm.AR('log_vol', rho=[rho], sigma=sigma_vol, shape=n_obs)
        
        # Observed returns
        obs = pm.Normal('obs', mu=0, sigma=pm.math.exp(log_vol/2), observed=returns)
        
        # Sample
        trace = pm.sample(
            draws=500,
            tune=500,
            chains=2,
            target_accept=target_accept,
            random_seed=42,
            return_inferencedata=True
        )
        
        results[target_accept] = trace

# Compare diagnostics
print("\n" + "="*60)
print("COMPARISON: Effect of target_accept")
print("="*60)

for target_accept, trace in results.items():
    n_divergences = trace.sample_stats.diverging.sum().values
    mean_accept = trace.sample_stats.acceptance_rate.mean().values
    
    print(f"\ntarget_accept={target_accept}:")
    print(f"  Divergences: {n_divergences}")
    print(f"  Mean acceptance rate: {mean_accept:.3f}")
    print(f"  ESS (sigma_vol): {az.ess(trace, var_names=['sigma_vol']).sigma_vol.values:.0f}")

## 4. Diagnosing NUTS Problems

### Common Issues

1. **Divergences**: Indicate the geometry is too complex for current step size
2. **Low E-BFMI**: Energy Bayesian Fraction of Missing Information < 0.3
3. **High Rhat**: Rhat > 1.01 indicates chains haven't converged
4. **Low ESS**: Effective sample size < 400 per chain
5. **Max treedepth warnings**: Need longer trajectories

Let's create a pathological model and diagnose it.

In [ ]:
# Purpose: Demonstrate diagnostic workflow for problematic model
# Key Concept: ArviZ provides comprehensive diagnostics

# Neal's funnel - a classic challenging geometry
with pm.Model() as funnel:
    # Top of funnel (wide)
    y = pm.Normal('y', mu=0, sigma=3)
    
    # Bottom of funnel (narrow when y is small)
    x = pm.Normal('x', mu=0, sigma=pm.math.exp(y/2), shape=5)
    
    # Sample with default settings (will have issues)
    trace_bad = pm.sample(
        draws=500,
        tune=500,
        chains=2,
        random_seed=42,
        return_inferencedata=True
    )

# Check diagnostics
print("\n" + "="*60)
print("DIAGNOSTIC REPORT")
print("="*60)

# 1. Divergences
n_divergences = trace_bad.sample_stats.diverging.sum().values
print(f"\n1. Divergences: {n_divergences}")
if n_divergences > 0:
    print("   ⚠️ PROBLEM: Divergences detected!")
    print("   → Try increasing target_accept to 0.95 or 0.99")

# 2. E-BFMI
ebfmi = az.bfmi(trace_bad)
print(f"\n2. E-BFMI: {ebfmi.values.mean():.3f}")
if ebfmi.values.mean() < 0.3:
    print("   ⚠️ PROBLEM: Low E-BFMI!")
    print("   → Model may be misspecified or needs reparameterization")

# 3. Rhat
rhat = az.rhat(trace_bad)
max_rhat = max(rhat.max().values())
print(f"\n3. Max Rhat: {max_rhat:.4f}")
if max_rhat > 1.01:
    print("   ⚠️ PROBLEM: Chains haven't converged!")
    print("   → Run longer chains or increase target_accept")

# 4. ESS
ess = az.ess(trace_bad)
min_ess = min([ess[var].min().values for var in ess.data_vars])
print(f"\n4. Min ESS: {min_ess:.0f}")
if min_ess < 400:
    print("   ⚠️ PROBLEM: Low effective sample size!")
    print("   → Need more samples or better mixing")

print("\n" + "="*60)

# Visualize the problem
az.plot_pair(
    trace_bad,
    var_names=['y', 'x'],
    coords={'x_dim_0': [0]},  # Just first x dimension
    divergences=True,
    figsize=(10, 8)
)
plt.suptitle('Neal\'s Funnel: Divergences Shown in Red', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 5. Fixing Sampling Issues: Reparameterization

The **non-centered parameterization** is a powerful technique for resolving funnel geometries.

Instead of:
```
x ~ Normal(0, exp(y/2))
```

We use:
```
x_raw ~ Normal(0, 1)
x = x_raw * exp(y/2)
```

This breaks the dependence in the posterior geometry.

In [ ]:
# Purpose: Demonstrate non-centered parameterization
# Key Concept: Reparameterization can eliminate sampling issues

# Neal's funnel - reparameterized
with pm.Model() as funnel_fixed:
    # Top of funnel
    y = pm.Normal('y', mu=0, sigma=3)
    
    # Non-centered parameterization
    x_raw = pm.Normal('x_raw', mu=0, sigma=1, shape=5)
    x = pm.Deterministic('x', x_raw * pm.math.exp(y/2))
    
    # Sample with same settings
    trace_fixed = pm.sample(
        draws=500,
        tune=500,
        chains=2,
        random_seed=42,
        return_inferencedata=True
    )

# Compare diagnostics
print("\n" + "="*60)
print("BEFORE vs AFTER Reparameterization")
print("="*60)

print("\nCentered parameterization:")
print(f"  Divergences: {trace_bad.sample_stats.diverging.sum().values}")
print(f"  E-BFMI: {az.bfmi(trace_bad).values.mean():.3f}")

print("\nNon-centered parameterization:")
print(f"  Divergences: {trace_fixed.sample_stats.diverging.sum().values}")
print(f"  E-BFMI: {az.bfmi(trace_fixed).values.mean():.3f}")

print("\n✅ Non-centered parameterization resolved the issues!")

# Visualize improvement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
az.plot_trace(trace_bad, var_names=['y'], axes=axes[0:1])
axes[0].set_title('Centered (Problematic)')

# After  
az.plot_trace(trace_fixed, var_names=['y'], axes=axes[1:2])
axes[1].set_title('Non-centered (Fixed)')

plt.tight_layout()
plt.show()

## 6. Application: Commodity Price Model with NUTS

Let's apply NUTS to a realistic commodity forecasting model: a hierarchical stochastic volatility model for multiple crude oil grades.

In [ ]:
# Purpose: Build hierarchical SV model for multiple commodities
# Key Concept: NUTS handles complex hierarchical models efficiently

# Generate synthetic data for 3 oil grades
np.random.seed(456)
n_obs = 150
n_commodities = 3
commodity_names = ['WTI', 'Brent', 'Dubai']

# True parameters (shared volatility dynamics)
true_rho = 0.92
true_sigma_vol = 0.25

# Generate returns for each commodity
returns_dict = {}
for i, name in enumerate(commodity_names):
    # Generate log-volatility
    log_vol = np.zeros(n_obs)
    log_vol[0] = np.random.randn() * 0.5
    for t in range(1, n_obs):
        log_vol[t] = true_rho * log_vol[t-1] + np.random.randn() * true_sigma_vol
    
    # Generate returns
    returns_dict[name] = np.random.randn(n_obs) * np.exp(log_vol / 2) * (1 + i * 0.1)

# Stack into array
returns_array = np.column_stack([returns_dict[name] for name in commodity_names])

print(f"Generated {n_obs} days of returns for {n_commodities} commodities")
print(f"\nReturn statistics:")
for name in commodity_names:
    r = returns_dict[name]
    print(f"  {name}: mean={r.mean():.4f}, std={r.std():.4f}")

In [ ]:
# Purpose: Fit hierarchical SV model with NUTS
# Key Concept: Use non-centered parameterization for hierarchical models

with pm.Model() as hierarchical_sv:
    # Hyperpriors for volatility process (shared across commodities)
    mu_rho = pm.Beta('mu_rho', alpha=20, beta=2)  # Mean persistence
    sigma_rho = pm.Exponential('sigma_rho', 10)   # Variation in persistence
    
    mu_sigma_vol = pm.Exponential('mu_sigma_vol', 2)  # Mean volatility of volatility
    
    # Commodity-specific parameters (non-centered)
    rho_raw = pm.Normal('rho_raw', mu=0, sigma=1, shape=n_commodities)
    rho = pm.Deterministic('rho', mu_rho + rho_raw * sigma_rho)
    
    sigma_vol = pm.Exponential('sigma_vol', 1/mu_sigma_vol, shape=n_commodities)
    
    # Log-volatility process for each commodity (non-centered)
    log_vol_raw = pm.Normal('log_vol_raw', mu=0, sigma=1, shape=(n_obs, n_commodities))
    
    # Transform to centered parameterization
    log_vol = pm.Deterministic(
        'log_vol',
        log_vol_raw * sigma_vol[None, :]  # Broadcasting
    )
    
    # Observed returns
    obs = pm.Normal(
        'obs',
        mu=0,
        sigma=pm.math.exp(log_vol / 2),
        observed=returns_array
    )
    
    # Sample with NUTS (increased target_accept for stability)
    trace_hsv = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.95,
        random_seed=42,
        return_inferencedata=True
    )

# Check diagnostics
print("\n" + "="*60)
print("MODEL DIAGNOSTICS")
print("="*60)

n_divergences = trace_hsv.sample_stats.diverging.sum().values
print(f"\nDivergences: {n_divergences}")

summary = az.summary(trace_hsv, var_names=['mu_rho', 'mu_sigma_vol', 'rho', 'sigma_vol'])
print("\nParameter estimates:")
print(summary[['mean', 'sd', 'r_hat', 'ess_bulk']])

print(f"\nTrue mu_rho: {true_rho:.3f}")
print(f"True mu_sigma_vol: {true_sigma_vol:.3f}")

In [ ]:
# Purpose: Visualize posterior and diagnostics
# Key Concept: Good diagnostics indicate reliable inference

# Trace plots for hyperparameters
az.plot_trace(
    trace_hsv,
    var_names=['mu_rho', 'mu_sigma_vol'],
    figsize=(12, 6)
)
plt.suptitle('Hierarchical SV Model: Hyperparameters', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

# Posterior predictive check
with hierarchical_sv:
    ppc = pm.sample_posterior_predictive(trace_hsv, random_seed=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, name in enumerate(commodity_names):
    axes[i].hist(
        ppc.posterior_predictive['obs'].values[:, :, :, i].flatten(),
        bins=30,
        alpha=0.5,
        label='Posterior predictive',
        density=True
    )
    axes[i].hist(
        returns_dict[name],
        bins=30,
        alpha=0.5,
        label='Observed',
        density=True
    )
    axes[i].set_title(name)
    axes[i].legend()
    axes[i].set_xlabel('Returns')

plt.suptitle('Posterior Predictive Check', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print("✅ Model fits observed data well!")

## 7. Exercises

### Exercise 6.3.1: Tune NUTS for a Challenging Model

**Task:** Fit the following model with minimal divergences by tuning NUTS parameters.

**Expected Output:** Less than 5 divergences and all Rhat < 1.01

In [ ]:
# Exercise setup: Generate data from Student-t model (heavy tails)
np.random.seed(789)
exercise_data = stats.t.rvs(df=3, loc=0, scale=2, size=100)

# YOUR CODE HERE
# ---------------
# Task: Fit a Student-t model and tune NUTS to minimize divergences
# Hints:
# 1. Start with target_accept=0.95
# 2. You may need to increase max_treedepth
# 3. Use more tune samples (e.g., 1500)

with pm.Model() as student_t_model:
    # Define model
    # mu = ...
    # sigma = ...
    # nu = ...
    # obs = pm.StudentT('obs', nu=nu, mu=mu, sigma=sigma, observed=exercise_data)
    
    # Sample with tuned parameters
    # trace_exercise = pm.sample(...)
    pass

# trace_exercise = None  # Replace with your trace

# Solution will be provided below

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_6_3_1():
    assert 'trace_exercise' in globals(), "Create trace_exercise variable"
    assert trace_exercise is not None, "trace_exercise should not be None"
    
    # Check divergences
    n_div = trace_exercise.sample_stats.diverging.sum().values
    assert n_div < 5, f"Too many divergences ({n_div}). Try increasing target_accept."
    
    # Check Rhat
    rhat = az.rhat(trace_exercise)
    max_rhat = max(rhat.max().values())
    assert max_rhat < 1.01, f"Rhat too high ({max_rhat:.4f}). Chains haven't converged."
    
    print("✅ Exercise 6.3.1 passed!")
    print(f"   Divergences: {n_div}")
    print(f"   Max Rhat: {max_rhat:.4f}")

# Uncomment to test:
# test_exercise_6_3_1()

<details>
<summary>Click for Exercise 6.3.1 Solution</summary>

```python
with pm.Model() as student_t_model:
    mu = pm.Normal('mu', mu=0, sigma=10)
    sigma = pm.Exponential('sigma', 1)
    nu = pm.Gamma('nu', alpha=2, beta=0.1)  # Prior for degrees of freedom
    
    obs = pm.StudentT('obs', nu=nu, mu=mu, sigma=sigma, observed=exercise_data)
    
    trace_exercise = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        target_accept=0.95,
        max_treedepth=12,
        random_seed=42,
        return_inferencedata=True
    )
```
</details>

### Exercise 6.3.2: Reparameterize a Hierarchical Model

**Task:** Convert the centered parameterization below to non-centered.

**Expected Output:** Reduced divergences and improved ESS

In [ ]:
# Exercise setup: Centered parameterization (problematic)
np.random.seed(101)
n_groups = 5
n_per_group = 20
group_means = np.random.randn(n_groups) * 2
exercise_data_hier = np.concatenate([
    np.random.randn(n_per_group) + group_means[i] 
    for i in range(n_groups)
])
group_idx = np.repeat(np.arange(n_groups), n_per_group)

# Centered version (will have issues)
with pm.Model() as centered_hier:
    mu_global = pm.Normal('mu_global', mu=0, sigma=10)
    sigma_global = pm.Exponential('sigma_global', 1)
    
    # Centered parameterization
    mu_group = pm.Normal('mu_group', mu=mu_global, sigma=sigma_global, shape=n_groups)
    
    sigma = pm.Exponential('sigma', 1)
    obs = pm.Normal('obs', mu=mu_group[group_idx], sigma=sigma, observed=exercise_data_hier)
    
    trace_centered = pm.sample(500, tune=500, chains=2, random_seed=42, return_inferencedata=True)

print(f"Centered divergences: {trace_centered.sample_stats.diverging.sum().values}")

# YOUR CODE HERE
# ---------------
# Task: Convert to non-centered parameterization
# Hint: mu_group = mu_global + mu_group_raw * sigma_global

# with pm.Model() as noncentered_hier:
#     ...
#     trace_noncentered = pm.sample(...)

# trace_noncentered = None  # Replace with your trace

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_6_3_2():
    assert 'trace_noncentered' in globals(), "Create trace_noncentered variable"
    assert trace_noncentered is not None, "trace_noncentered should not be None"
    
    # Check improvement in divergences
    n_div_centered = trace_centered.sample_stats.diverging.sum().values
    n_div_noncentered = trace_noncentered.sample_stats.diverging.sum().values
    
    print(f"Centered divergences: {n_div_centered}")
    print(f"Non-centered divergences: {n_div_noncentered}")
    
    assert n_div_noncentered < n_div_centered, "Non-centered should have fewer divergences"
    
    # Check ESS improvement
    ess_centered = az.ess(trace_centered, var_names=['mu_global']).mu_global.values
    ess_noncentered = az.ess(trace_noncentered, var_names=['mu_global']).mu_global.values
    
    print(f"Centered ESS: {ess_centered:.0f}")
    print(f"Non-centered ESS: {ess_noncentered:.0f}")
    
    print("\n✅ Exercise 6.3.2 passed!")

# Uncomment to test:
# test_exercise_6_3_2()

<details>
<summary>Click for Exercise 6.3.2 Solution</summary>

```python
with pm.Model() as noncentered_hier:
    mu_global = pm.Normal('mu_global', mu=0, sigma=10)
    sigma_global = pm.Exponential('sigma_global', 1)
    
    # Non-centered parameterization
    mu_group_raw = pm.Normal('mu_group_raw', mu=0, sigma=1, shape=n_groups)
    mu_group = pm.Deterministic('mu_group', mu_global + mu_group_raw * sigma_global)
    
    sigma = pm.Exponential('sigma', 1)
    obs = pm.Normal('obs', mu=mu_group[group_idx], sigma=sigma, observed=exercise_data_hier)
    
    trace_noncentered = pm.sample(
        500, tune=500, chains=2, random_seed=42, return_inferencedata=True
    )
```
</details>

### Exercise 6.3.3: Interpret NUTS Diagnostics

**Task:** Run the model below and diagnose what's wrong. Then propose a fix.

**Expected Output:** Identification of the issue and a working solution

In [ ]:
# Exercise setup: Problematic model
np.random.seed(202)
exercise_data_diag = np.random.exponential(2, size=50)

with pm.Model() as problematic_model:
    # This prior is too informative/wrong
    rate = pm.Normal('rate', mu=10, sigma=0.1)  # Problem: rate should be positive!
    obs = pm.Exponential('obs', lam=rate, observed=exercise_data_diag)
    
    trace_problem = pm.sample(500, tune=500, chains=2, random_seed=42, return_inferencedata=True)

# YOUR CODE HERE
# ---------------
# 1. Examine diagnostics for trace_problem
# 2. Identify what's wrong
# 3. Create a fixed model

# What's wrong? Write your diagnosis:
diagnosis = ""  # YOUR ANSWER HERE

# with pm.Model() as fixed_model:
#     ...
#     trace_fixed_diag = pm.sample(...)

# trace_fixed_diag = None  # Replace with your trace

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_6_3_3():
    assert diagnosis != "", "Write your diagnosis"
    assert 'positive' in diagnosis.lower() or 'exponential' in diagnosis.lower(), \
        "Diagnosis should mention that rate must be positive"
    
    assert 'trace_fixed_diag' in globals(), "Create trace_fixed_diag variable"
    assert trace_fixed_diag is not None, "trace_fixed_diag should not be None"
    
    # Check that fixed model works better
    n_div = trace_fixed_diag.sample_stats.diverging.sum().values
    assert n_div == 0, f"Fixed model still has divergences ({n_div})"
    
    print("✅ Exercise 6.3.3 passed!")
    print(f"Your diagnosis: {diagnosis}")

# Uncomment to test:
# test_exercise_6_3_3()

<details>
<summary>Click for Exercise 6.3.3 Solution</summary>

```python
diagnosis = "The rate parameter must be positive, but Normal prior allows negative values. This causes the exponential likelihood to be undefined for negative rates, leading to sampling issues."

with pm.Model() as fixed_model:
    # Use a prior that enforces positivity
    rate = pm.Exponential('rate', 1)  # Or Gamma, HalfNormal, etc.
    obs = pm.Exponential('obs', lam=rate, observed=exercise_data_diag)
    
    trace_fixed_diag = pm.sample(
        500, tune=500, chains=2, random_seed=42, return_inferencedata=True
    )
```
</details>

---

## Summary

### Key Takeaways

1. **NUTS is adaptive**: Automatically tunes trajectory length using the no-U-turn criterion
2. **Key parameters**: `target_accept` (0.8-0.99) and `max_treedepth` (default 10)
3. **Common issues**: Divergences, low E-BFMI, high Rhat, low ESS, max treedepth warnings
4. **Reparameterization**: Non-centered parameterization resolves funnel geometries
5. **Diagnostics**: ArviZ provides comprehensive tools for checking convergence
6. **Tuning strategy**: Start with target_accept=0.95 for challenging models
7. **Commodity models**: NUTS handles hierarchical stochastic volatility models effectively

### What's Next

- **Next notebook**: Variational Inference in PyMC (faster approximate inference)
- **Advanced topic**: Custom step methods and Gibbs sampling
- **Application**: Production-scale Bayesian forecasting pipelines

### Additional Resources

- Hoffman & Gelman (2014): "The No-U-Turn Sampler"
- Betancourt (2017): "A Conceptual Introduction to Hamiltonian Monte Carlo"
- PyMC documentation: https://www.pymc.io/projects/docs/en/stable/api/samplers.html
- ArviZ diagnostics guide: https://arviz-devs.github.io/arviz/

### Quick Reference: NUTS Tuning

| Issue | Solution |
|-------|----------|
| Divergences | Increase `target_accept` to 0.95 or 0.99 |
| Low E-BFMI | Check model specification, consider reparameterization |
| High Rhat | Run longer chains, check for multimodality |
| Low ESS | More samples or reparameterization |
| Max treedepth | Increase `max_treedepth` to 12 or 15 |
| Funnel geometry | Use non-centered parameterization |